In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# ================= CRITICAL PATCH START =================
# We must define the dummy callback and force it into the N2V module
# BEFORE we initialize or run the model.

class DummyCARETensorBoardImage(tf.keras.callbacks.Callback):
    def __init__(self, model, data, log_dir, n_images, prob_out):
        super().__init__() 
    def on_epoch_end(self, epoch, logs=None):
        pass # Do nothing

# 1. Import the internal module where N2V is defined
import n2v.models.n2v_standard

# 2. Overwrite the class reference INSIDE that module
n2v.models.n2v_standard.CARETensorBoardImage = DummyCARETensorBoardImage
print("Patched CARETensorBoardImage in n2v_standard")
# ================= CRITICAL PATCH END ===================

from n2v.models import N2VConfig, N2V
from csbdeep.utils import plot_history

# ================= CONFIGURATION =================
TRAIN_PATH = os.path.expandvars("$VF_DENOISE_ROOT/dataset_train_val_test/VAE/fine_tune_dataset_Nge1_train.npz")
VAL_PATH   = os.path.expandvars("$VF_DENOISE_ROOT/dataset_train_val_test/VAE/fine_tune_dataset_Nge1_val.npz")

MODEL_NAME = 'n2v_vf_8x9'
BASE_DIR   = 'models'

# ================= MAPPING & PADDING LOGIC =================
def get_valid_indices_8x9():
    nulls = [0,1,2,7,8,9,10,17,18,34,43,45,54,55,62,63,64,65,70,71]
    all_indices = np.arange(8 * 9)
    valid = [i for i in all_indices if i not in nulls]
    return valid

def vector_to_img_8x9(vectors):
    """ Converts (N, 52) -> (N, 8, 9, 1) """
    N = vectors.shape[0]
    H, W = 8, 9
    images = np.zeros((N, H * W), dtype=np.float32)
    valid_indices = get_valid_indices_8x9()
    limit = min(len(valid_indices), vectors.shape[1])
    images[:, valid_indices[:limit]] = vectors[:, :limit]
    return images.reshape(N, H, W, 1)

def pad_to_divisible(data):
    """ Pads (N, 8, 9, 1) to (N, 8, 12, 1) for U-Net compatibility. """
    return np.pad(data, ((0,0), (0,0), (0,3), (0,0)), mode='constant')

def load_and_pad(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Path not found: {path}")

    d = np.load(path)
    if 'td' in d: raw = d['td']
    elif 'data' in d: raw = d['data']
    else: raise ValueError(f"Unknown keys: {list(d.keys())}")
    
    # 1. Map to 8x9
    imgs = vector_to_img_8x9(raw)
    
    # 2. Pad to 8x12
    imgs_padded = pad_to_divisible(imgs)
    return imgs_padded

# ================= EXECUTION =================
print(f"Loading Data...")
X_train = load_and_pad(TRAIN_PATH)
X_val   = load_and_pad(VAL_PATH)
print(f"Data Loaded (Padded): {X_train.shape} | {X_val.shape}")

# Config
config = N2VConfig(
    X_train, 
    unet_kern_size=3,
    unet_n_first=32,      
    unet_n_depth=2,       
    train_steps_per_epoch=int(X_train.shape[0]/128),
    train_epochs=100,
    train_loss='mse',
    batch_norm=True,
    train_batch_size=128,
    n2v_perc_pix=1.5,     
    n2v_patch_shape=(8, 12),
    n2v_manipulator='uniform_withCP',
    n2v_neighborhood_radius=2,
    train_tensorboard=False
)

# 2. Fix for .weights.h5 error
config.train_checkpoint = "weights_best.weights.h5"

# Train
print("Starting Training...")
model = N2V(config, MODEL_NAME, basedir=BASE_DIR)
history = model.train(X_train, X_val)

# Plot
plot_history(history, ['loss', 'val_loss'])
plt.savefig(f"{BASE_DIR}/{MODEL_NAME}/history.png")
plt.show()

In [ ]:
import os
from pathlib import Path
DATA_ROOT = Path(os.environ.get("VF_DATA_ROOT", "/path/to/vf_oct_pairs"))   # see config/env.example.sh

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from n2v.models import N2V
from matplotlib import colors
from mpl_toolkits.axes_grid1 import make_axes_locatable
from pathlib import Path
import random

# ================= CONFIGURATION =================
MODEL_NAME = 'n2v_vf_8x9'
BASE_DIR   = 'models'
# Test data folder
DATA_FOLDER = DATA_ROOT / "original_below350"

# ================= 1. PLOTTING TOOLS =================
def gen_vfmat(tds):
    """Maps 52 points to 8x9 grid (72 pixels) with NaNs for nulls."""
    mat = np.full((8, 9), np.nan)
    # 8x9 grid mask indices (24-2 layout)
    nulls = [0,1,2,7,8,9,10,17,18,34,43,45,54,55,62,63,64,65,70,71]
    
    k = 0
    for i in range(8):
        for j in range(9):
            pos = i * 9 + j
            if pos not in nulls:
                if k < len(tds):
                    mat[i][j] = tds[k]
                    k += 1
    return mat

def plot_before_after(noisy, denoised, filename):
    """Plots 3 panels: Noisy, Denoised, Difference."""
    
    # Prepare Matrices
    mat_noisy = gen_vfmat(noisy)
    mat_denoised = gen_vfmat(denoised)
    mat_diff = mat_noisy - mat_denoised  # "What did N2V remove?"
    
    # Setup Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    
    titles = [
        "Original Input (Noisy)",
        "N2V Output (Denoised)",
        "Difference (Noise Removed)"
    ]
    mats = [mat_noisy, mat_denoised, mat_diff]
    
    # Colorbar Settings
    # VF values usually -30 to +10 dB
    divnorm = colors.TwoSlopeNorm(vmin=-35, vcenter=0, vmax=15)
    # Diff map usually -5 to +5 dB
    divnorm_diff = colors.TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10)

    for i, ax in enumerate(axes):
        ax.axis('off')
        
        # Use Red-Blue for Difference, Blue-White-Red for VFs
        cmap = 'RdBu_r' if i == 2 else 'bwr_r'
        norm = divnorm_diff if i == 2 else divnorm
        
        im = ax.imshow(mats[i], cmap=cmap, norm=norm, aspect='auto')
        ax.set_title(titles[i], fontsize=14, fontweight='bold')
        
        # Add Text Values
        for (r, c), z in np.ndenumerate(mats[i]):
            if not np.isnan(z):
                # White text for dark colors, Black for light
                color = 'white' if abs(z) > 20 or (i==2 and abs(z)>5) else 'black'
                ax.text(c, r, f"{z:.1f}", ha='center', va='center', fontsize=9, color=color)

        # Add Colorbar
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("bottom", size="5%", pad=0.1)
        plt.colorbar(im, cax=cax, orientation="horizontal")
    
    plt.suptitle(f"File: {filename}", fontsize=16)
    plt.show()

# ================= 2. N2V INFERENCE LOGIC =================
def get_valid_indices_8x9():
    nulls = [0,1,2,7,8,9,10,17,18,34,43,45,54,55,62,63,64,65,70,71]
    return [i for i in np.arange(72) if i not in nulls]

def predict_single_vector(model, vector):
    # A. Vector -> 8x9 Image (8, 9, 1)
    img_flat = np.zeros(72, dtype=np.float32)
    valid_indices = get_valid_indices_8x9()
    img_flat[valid_indices] = vector
    img_8x9 = img_flat.reshape(8, 9, 1)
    
    # B. Pad -> 8x12 (Divisible by 4)
    img_padded = np.pad(img_8x9, ((0,0), (0,3), (0,0)), mode='constant')
    
    # C. Predict
    # FIX: Pass 3D array directly. Do NOT add [np.newaxis, ...].
    # Input shape: (8, 12, 1). Axes: 'YXC' (Length 3). Matches!
    pred_padded = model.predict(img_padded, axes='YXC')
    
    # D. Crop -> 8x9
    pred_8x9 = pred_padded[:, :9, :]
    
    # E. Image -> Vector
    return pred_8x9.flatten()[valid_indices]

# ================= 3. MAIN =================
def main():
    # 1. Patch N2V (AttributeError Fix)
    import csbdeep.utils.tf
    class Dummy(tf.keras.callbacks.Callback): pass
    csbdeep.utils.tf.CARETensorBoardImage = Dummy
    
    # 2. Load Model
    print(f"Loading Model: {MODEL_NAME}")
    model = N2V(config=None, name=MODEL_NAME, basedir=BASE_DIR)
    
    # 3. Get Files
    all_files = list(DATA_FOLDER.glob("*.npz"))
    print(f"Found {len(all_files)} files in {DATA_FOLDER}")
    
    # 4. Visualize Random Samples
    num_samples = 5
    selected_files = random.sample(all_files, num_samples)
    
    print("Generating Plots...")
    for f_path in selected_files:
        try:
            data = np.load(f_path, allow_pickle=True)
            vec_noisy = data['td'].astype(np.float32)
            
            # Run Inference
            vec_denoised = predict_single_vector(model, vec_noisy)
            
            # Plot
            plot_before_after(vec_noisy, vec_denoised, f_path.name)
            
        except Exception as e:
            print(f"Skipping {f_path.name}: {e}")

if __name__ == "__main__":
    main()

# nohup python3 inference_vf.py > vf_inference.log 2>&1 &